# Parte 2: Engenharia de Atributos e Preparação de Dados

Nesta fase da nossa história, transformamos dados transacionais brutos em métricas de comportamento: **Recência, Frequência e Valor Monetário (RFM)**. Também preparamos o terreno para o algoritmo, tratando valores discrepantes e normalizando as escalas.

### O que é RFM?
- **Recência (R)**: Dias desde a última compra do cliente.
- **Frequência (F)**: Quantidade total de compras realizadas.
- **Valor Monetário (M)**: Valor total gasto pelo cliente.

In [ ]:
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

# Carregando os dados pré-processados
df = pd.read_csv('../data/df_clientes_header.csv')


### 1. Cálculo do RFM por Cliente

In [ ]:
df_rfm = df.rename(columns={
    'valor_total': 'Monetario',
    'qtde_compras': 'Frequencia'
})

data_hoje = pd.to_datetime('2026-06-01')

df_rfm['Recencia'] = (data_hoje - pd.to_datetime(df_rfm['data_da_ultima_compra'])).dt.days

df_rfm = df_rfm[['cliente_id','Recencia', 'Frequencia', 'Monetario']]
df_rfm['cliente_id'] = df_rfm['cliente_id'].replace('CUST-', '', regex=True).astype({'cliente_id': int})
df_rfm = df_rfm.set_index('cliente_id')   

df_rfm.head()

### 2. Tratamento de Outliers (IQR)

Para que os nossos grupos não sejam distorcidos por clientes com comportamentos extremos (como grandes atacadistas), utilizamos o método do Intervalo Interquartil para limpar a base.

In [ ]:
def remover_outliers_iqr(df, colunas):
    df_clean = df.copy()
    for col in colunas:
        Q1 = df_clean[col].quantile(0.25)
        Q3 = df_clean[col].quantile(0.75)
        IQR = Q3 - Q1
        limite_inferior = Q1 - 1.5 * IQR
        limite_superior = Q3 + 1.5 * IQR
        df_clean = df_clean[(df_clean[col] >= limite_inferior) & (df_clean[col] <= limite_superior)]
    return df_clean

colunas_rfm = ['Recencia', 'Frequencia', 'Monetario']
df_rfm_clean = remover_outliers_iqr(df_rfm, colunas_rfm)

print(f"Registros antes: {len(df_rfm)} | Registros depois: {len(df_rfm_clean)}")

### 3. Escalonamento (Feature Scaling)

Como o K-Means utiliza distância euclidiana, precisamos garantir que o Valor Monetário (em reais) não domine a Recência (em dias). O `StandardScaler` coloca todas as variáveis na mesma ordem de grandeza.

In [ ]:
scaler = StandardScaler()
df_rfm_scaled = scaler.fit_transform(df_rfm_clean)

df_rfm_scaled = pd.DataFrame(
    df_rfm_scaled, 
    columns=df_rfm_clean.columns, 
    index=df_rfm_clean.index
)

print("Dados prontos para a modelagem!")
df_rfm_scaled.head()

### Salvando o progresso
Exportamos os dados processados para que o próximo capítulo da nossa história possa começar.

In [ ]:
df_rfm_clean.to_csv('../data/df_rfm_final.csv')
df_rfm_scaled.to_csv('../data/df_rfm_scaled.csv')
print("Arquivos exportados com sucesso.")